# Capítulo 14 — RAG Odontológico (Retrieval-Augmented Generation)

Este notebook demonstra um sistema simplificado de RAG (Retrieval-Augmented
Generation) para responder perguntas odontológicas com base em literatura
científica indexada.

In [ ]:
# ============================================================
# 1. Configuração
# ============================================================
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Módulos carregados!")

In [ ]:
# ============================================================
# 2. Base de conhecimento odontológica
# ============================================================
documentos = [
    # Documentos sobre cárie dentária
    """
    A cárie dentária é uma doença multifatorial causada pela desmineralização
    dos tecidos dentários por ácidos produzidos por bactérias do biofilme
    dental, principalmente Streptococcus mutans. O diagnóstico precoce pode
    ser realizado através de inspeção visual, radiografia interproximal e
    dispositivos de fluorescência laser.
    """,
    
    # Documentos sobre periodontite
    """
    A periodontite é uma doença inflamatória crônica dos tecidos de suporte
    dos dentes, causada por microorganismos periodontopatogênicos. O
    diagnóstico utiliza parâmetros como profundidade de sondagem, nível
    de inserção clínica, sangramento à sondagem e avaliação radiográfica
    da perda óssea. A classificação AAP/EFP 2018 define 4 estágios.
    """,
    
    # Documentos sobre radiografias panorâmicas
    """
    A radiografia panorâmica (ortopantomografia) é um exame de imagem
    extraoral que fornece uma visão geral da arcada dentária, seios
    maxilares e ATM. É amplamente utilizada em diagnóstico odontológico
    por sua baixa dose de radiação e ampla cobertura anatômica.
    """,
    
    # Documentos sobre IA em odontologia
    """
    A inteligência artificial tem sido aplicada na odontologia para
    diagnóstico auxiliado por computador (CAD), segmentação de imagens,
    detecção de cáries e lesões periapicais, classificação de estágios
    periodontais e planejamento de tratamento. Modelos de deep learning
    como CNNs e U-Net são os mais utilizados para análise de imagens.
    """,
    
    # Documentos sobre lesões bucais
    """
    Lesões bucais incluem uma variedade de condições que afetam a mucosa
    oral, incluindo úlceras aftosas, leucoplasia, eritroplasia, líquen
    plano oral e carcinomas. O diagnóstico diferencial é crucial e
    frequentemente requer biópsia para confirmação histopatológica.
    """,
    
    # Documentos sobre implantes dentários
    """
    Os implantes dentários são dispositivos de titânio inseridos no osso
    maxilar ou mandibular para substituir raízes dentárias perdidas.
    A osseointegração é o processo de fixação biológica entre o implante
    e o tecido ósseo, essencial para o sucesso a longo prazo.
    """,
]

titulos = [
    "Cárie Dentária",
    "Periodontite",
    "Radiografia Panorâmica",
    "IA em Odontologia",
    "Lesões Bucais",
    "Implantes Dentários",
]

print(f'Base de conhecimento com {len(documentos)} documentos carregada.')

In [ ]:
# ============================================================
# 3. Indexação TF-IDF
# ============================================================
vectorizer = TfidfVectorizer(
    max_df=0.8,
    min_df=1,
    stop_words='english',  # para demo; em produção usar português
    ngram_range=(1, 2),
)

# Remove acentos e normaliza para minúsculo
documentos_clean = [d.lower().strip() for d in documentos]
tfidf_matrix = vectorizer.fit_transform(documentos_clean)

print(f'Matriz TF-IDF: {tfidf_matrix.shape}')
print(f'Vocabulário: {len(vectorizer.get_feature_names_out())} termos')

In [ ]:
# ============================================================
# 4. Sistema de Busca (Retrieval)
# ============================================================
def search_dental_knowledge(query, top_k=3):
    """Busca documentos odontológicos relevantes para a consulta."""
    query_vec = vectorizer.transform([query.lower()])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Índices dos top_k documentos mais similares
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        if similarities[idx] > 0:
            results.append({
                'title': titulos[idx],
                'score': similarities[idx],
                'content': documentos[idx].strip(),
            })
    
    return results

# Teste
query = "Como diagnosticar periodontite?"
results = search_dental_knowledge(query)

print(f'Consulta: {query}')
print('\nResultados:')
for r in results:
    print(f"  [{r['score']:.2f}] {r['title']}: {r['content'][:100]}...")

In [ ]:
# ============================================================
# 5. Geração de Resposta (RAG simplificado)
# ============================================================
def rag_answer(query):
    """Sistema RAG simplificado: busca + sumarização por template."""
    results = search_dental_knowledge(query, top_k=2)
    
    if not results:
        return "Desculpe, não encontrei informações na base de conhecimento."
    
    # Concatena os documentos recuperados como contexto
    context = ' '.join([r['content'] for r in results])
    
    # Template de resposta (em produção, aqui seria um LLM)
    resposta = f"""
    Pergunta: {query}
    
    Com base na literatura odontológica, encontramos informações relevantes:
    
    Fontes consultadas:
    """
    for r in results:
        resposta += f'    - {r["title"]} (relevância: {r["score"]:.1%})\n'
    
    resposta += f"""
    Síntese:
    {context[:500]}...
    """
    return resposta

# Teste com múltiplas perguntas
perguntas = [
    "O que é cárie dentária e como diagnosticar?",
    "Como a IA ajuda no diagnóstico odontológico?",
    "O que é uma radiografia panorâmica?",
]

for pergunta in perguntas:
    print('=' * 70)
    print(rag_answer(pergunta))
    print()

In [ ]:
# ============================================================
# 6. Métricas de avaliação do RAG
# ============================================================
def evaluate_retrieval(queries, relevant_docs):
    """Avalia a precisão do retrieval."""
    precisions = []
    recalls = []
    
    for query, relevant in zip(queries, relevant_docs):
        results = search_dental_knowledge(query, top_k=3)
        retrieved_titles = set(r['title'] for r in results)
        relevant_set = set(relevant)
        
        if len(retrieved_titles) > 0:
            prec = len(retrieved_titles & relevant_set) / len(retrieved_titles)
        else:
            prec = 0
        if len(relevant_set) > 0:
            rec = len(retrieved_titles & relevant_set) / len(relevant_set)
        else:
            rec = 0
        
        precisions.append(prec)
        recalls.append(rec)
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'f1': 2 * np.mean(precisions) * np.mean(recalls) / (np.mean(precisions) + np.mean(recalls) + 1e-7)
    }

# Avaliação com consultas de teste
test_queries = [
    "diagnóstico de periodontite",
    "cárie tratamento",
    "implantes osseointegração",
    "lesões bucais diagnóstico",
]
test_relevant = [
    ["Periodontite"],
    ["Cárie Dentária"],
    ["Implantes Dentários"],
    ["Lesões Bucais"],
]

eval_metrics = evaluate_retrieval(test_queries, test_relevant)
print("Métricas de Avaliação do Retrieval:")
for metric, value in eval_metrics.items():
    print(f'  {metric}: {value:.2%}')

---
**Conclusão:** O sistema RAG simplificado recupera conhecimento
odontológico relevante. Em produção, integrar com LLM e base
de literatura científica completa para respostas mais precisas.